# What does the model track well, and what does it track poorly?

**Question (TREE Q3.H1.E2, exploratory).** The registered Q3 measurements
average over all phases and transitions. This notebook slices those same
measurements by the structure we designed into the stories. The slices are
which emotions, which triple families, how far apart two emotions sit in
valence-arousal space, and what the model confuses with what. Idea credit:
Peyton Li.

**Status: exploratory, hypothesis-generating.** We named the slices and the
scoring conventions in TREE.md before any record file existed (commit
`dc23007`). There are no registered pass bars here, and nothing in this
notebook becomes a claim on its own. A promising slice first becomes a
registered measurement, and then has to survive a falsification pass.

**Key concepts.**
- *Emotion probe*: one emotion's direction in the model's residual stream. We
  build it from the mean activation over that emotion's stories, minus the
  mean over the whole pool of emotions it is compared against, scaled to unit
  length. Four sets of probes appear below. `corpus` holds 171 emotions, built
  from stories taken from the corpus. `selfgen` holds 12 emotions, built from
  stories the model wrote itself. `deepseek` holds the same 12, built from
  stories DeepSeek wrote. `random` holds 24 random directions.
- *The twelve designed emotions*: the 12 emotions the designed story triples
  draw from. The `selfgen` and `deepseek` sets carry exactly these twelve
  probes.
- *Centered cosine*: before we take the cosine between an activation and a
  probe, we subtract the story-set mean per (layer, probe). This is the
  registered scoring convention, shared bit-identically with the Q3 scorer.
  Figure axes give it as "centered-cosine units".
- *Rank of the tagged emotion, and top-1 rate*: for one story phase we average
  each probe's centered cosine over the phase's tokens. We then rank the
  tagged emotion against the other probes in its set. Rank 1 means the correct
  probe scored highest. The top-1 rate is the fraction of phases at rank 1.
  With twelve probes, guessing gives a top-1 rate of 1/12 and a median rank of
  about 6.5.
- *R1 anticipation lead*: for one transition, we take the incoming emotion's
  mean centered cosine over the W=16 tokens just before the phase boundary.
  From it we subtract that emotion's mean over the 16 tokens before those. A
  positive lead means the incoming emotion's signal is already rising before
  the text switches.
- *Cluster bootstrap over triple_id, 95% CI*: every confidence interval (CI)
  resamples whole triples rather than individual stories or phases. Stories
  generated from one designed triple share their source text, so they are not
  independent samples.
- *NRC VAD*: the National Research Council valence-arousal-dominance lexicon
  (v2.1), human ratings of each emotion word's valence (pleasant vs
  unpleasant) and arousal (intensity). It is the human-side ruler for how far
  apart two emotions sit affectively.
- *Layer slider*: every figure carries a slider over the six extracted layers
  (6, 15, 24, 33, 42, 51), defaulting to layer 33, the registered primary
  cell. No plot privileges a single layer.

**Index.**

Part 1 (TREE Q3.H1.E2):
1. S1: per-emotion tracking quality (which emotions the probes read well)
2. S2: designed triple families against the two registered measures (rank of
   the tagged emotion, R1 lead)
3. S3: affective distance (does tracking follow valence-arousal geometry?)
4. S4: confusion structure (graceful neighbours or unstructured failure?)
5. S5: position and non-affect cuts
6. Cross-condition check at the primary layer, and the Part 1 takeaway

Part 2 (TREE Q3.H1.E3):
7. S6: named confusions (what the model reports when it disagrees with the tag)
8. S7: boundary lag vs stable relabeling (when inside a phase it gets it right)
9. S8: probe geometry as a difficulty predictor, vs the VAD competitor

**Records.** `scripts/dump_q3_records.py` writes one record per story phase
and one per transition. Its conventions are bit-identical to the registered
scorer (`score_q3_gate_r1.py`), through the shared module
`emotion_vectors.q3_conventions`. Those conventions are centered cosine
(story-set mean, `norms_centered` denominator), SEQUENTIAL stories only, phase
means, and the W=16 boundary-referenced anticipation lead. The dumps also
carry per-phase token-count thirds (`phase_third_scores`), the records S7
needs. Each record
carries `triple_id`, the designed `category`, the tagged emotion(s), and
position.

| condition | stories written by | probes | role |
|---|---|---|---|
| `q3_records_it_v2` | Gemma-4-31B-it | corpus-171 post-fix + selfgen-12 + deepseek-12 + random-24 | primary |
| `q3_records_it` | Gemma-4-31B-it | pre-fix corpus-171 + selfgen-12 + random-24 | v1 continuity check |
| `q3_records_deepseek` | DeepSeek v4 Pro | same three true probe sets + random | quality-transfer condition |
| `q3_records_deepseek_constant` | DeepSeek v4 Pro | same | constant-emotion control |
| `q3_records_base` | Gemma-4-31B-it (read by the BASE model) | corpus-171 base-model probes + random-24 | reader-model condition |

**Named confounds (from the registration).** Triple families draw different
emotion pools, so family effects and emotion effects are partly confounded.
Per-emotion n differs, and phase length varies by position. Every slice
reports its n, and the valence-arousal cut (S3) is the one that separates
affective distance from family label. We report uncertainty as a cluster
bootstrap over `triple_id`, because stories from one triple share their
source text.

In [1]:
# this cell loads the record dumps, the VAD lexicon, and the designed-triples
# flag through the notebook-11 exhibit library; every section below is
# load-call-show over emotion_vectors.taxonomy_report
import numpy as np

from emotion_vectors import taxonomy_report as tr

LAYERS, PRIMARY_LAYER = tr.LAYERS, tr.PRIMARY_LAYER

# one shared generator, threaded through the section builders in notebook
# order (S2, S3, S4, S5) so every bootstrap CI reproduces exactly
RNG = np.random.default_rng(20260723)

ARMS = tr.load_arms(["it_v2", "it", "deepseek", "deepseek_constant", "base"])
for arm, records in ARMS.items():
    print(
        f"{arm}: {len(records['phases'])} phases, "
        f"{len(records['transitions'])} transitions, "
        f"{len(records['labels'])} probes"
    )

# NRC VAD lexicon (third-party license, populated manually) and the
# triple_id -> has_nonaffect map from the designed triples file
VAD = tr.load_vad()
HAS_NONAFFECT = tr.load_has_nonaffect()

it_v2: 8938 phases, 5934 transitions, 219 probes
it: 8902 phases, 5910 transitions, 207 probes
deepseek: 8498 phases, 5662 transitions, 219 probes
deepseek_constant: 314 phases, 208 transitions, 219 probes
base: 8938 phases, 5934 transitions, 195 probes


## S1. Which emotions does the model track well?

**Question: is tracking quality uniform across the twelve designed emotions,
or do a few emotions carry the average?**

**Method, plainly.** We take every story phase in the primary condition
(Gemma-written stories, v2 probes). For each phase we ask what rank the tagged
emotion's mean centered cosine gets among the probes in the set. Rank 1 means
the probe for the emotion the story actually expresses scored highest. Here we
split that rank by tagged emotion, separately for the two twelve-emotion probe
sets: the same stories, read with probes built from two different story
sources. Each emotion gets its top-1 rate, meaning how often its own probe
wins outright, and its median rank.

In [2]:
# this cell computes per-emotion top-1 rate and median rank for the self-generated
# and DeepSeek probe sets on the primary arm, one bar chart per probe set, layer slider
fig, S1 = tr.s1_top1_figure(ARMS)
fig.show()

<details><summary><b>How to read this figure</b></summary>

One bar is one tagged emotion in one set of probes. Its height is the top-1
rate: the fraction of phases tagged with that emotion where its own probe
out-scored the other 11 in the set. The left panel uses the probes built from
Gemma's own stories (`selfgen`), the right panel the probes built from
DeepSeek's stories (`deepseek`). Both panels score the same Gemma-written
stories, so a difference between panels comes from the probes and not from the
stories. The label above each bar gives that emotion's median rank ("med") and
its phase count n. Bars are sorted by top-1 rate at the currently selected
layer, so the left-to-right order changes as you scrub. The slider scrubs the
six extracted layers (6, 15, 24, 33, 42, 51) and defaults to 33, the registered
primary layer. A good result is a wall of similar tall bars, meaning every
emotion is readable; guessing would put every bar at the 1/12 line. A bad
result is a steep staircase, where a few emotions carry the aggregate while the
others sit near that line. An emotion with a low top-1 rate can still hold a
decent median rank, which the "med" label shows, so check both before calling
an emotion untracked. This figure has no error bars; the per-emotion n is
printed on each bar instead. Registered caveat: emotions are not balanced
across designed families (the family x emotion-pool confound) and per-emotion n
differs. A per-emotion difference can therefore reflect the story contexts an
emotion appears in rather than the emotion alone. Each set of probes also comes
from a different set of stories, so what we measure is probe quality times
expression strength, not a property of the model alone.

</details>

## S2. Which designed triple families are easy, which are hard?

**Question: does tracking difficulty follow the five families the triples
were designed in?**

**Method, plainly.** Peyton's triples were designed in five families:
**A_superposition** (blended states), **B_conflict** (same-valence
conflicting emotions, the hard discrimination case), **D_timescale** (mood vs
flash), **E_arousal_mismatch** (same valence, different arousal),
**F_valence_spread** (cross-valence, the designed easy case). We compute both
registered measures per family with the `selfgen` probes: the median rank of
the tagged emotion (identity) and the mean R1 lead (anticipation). We compute
each one twice, once on the Gemma-written stories and once on the
DeepSeek-written stories. Each carries a 95% cluster-bootstrap CI over
triples.

In [3]:
# this cell computes the tagged probe's median rank (read G) and the R1 mean lead per
# designed family, self-generated probes, primary arm vs deepseek-story arm, with
# cluster bootstrap CIs
fig, S2 = tr.s2_family_figure(ARMS, RNG)
fig.show()

<details><summary><b>How to read this figure</b></summary>

One bar is one designed family on one set of stories: blue is the
Gemma-written primary condition (`it_v2`), orange is the DeepSeek-written one,
both scored with the `selfgen` probes. Left panel: the family's median rank
for the tagged emotion across that family's phases. Lower is better, 1 is
perfect, and guessing sits near 6.5 of 12. Right panel: the family's mean R1
anticipation lead in centered-cosine units. Higher is better, and zero means
no pre-boundary rise. Error bars are 95% cluster-bootstrap CIs over
`triple_id`, resampling whole triples because stories from one triple share
their source text. A right-panel bar whose CI excludes zero marks a family
with a real anticipation lead in that set of stories. The slider scrubs the
six extracted layers (6, 15, 24, 33, 42, 51), default 33, and the title
restates that layer's own verdict. A good result for the design would be
family separation matching the intended difficulty ordering (valence spread
easiest, same-valence conflict hardest) with CIs that do not swallow the
differences. A bad result is flat bars, or CIs so wide that families are
indistinguishable. Registered caveat: each family draws a different emotion
pool, so a family effect here can be an emotion-composition effect in
disguise. S3 below is the cut designed to separate the affective-distance
construct from the family label.

</details>

## S3. Does tracking follow affective distance?

**Question: for a transition from emotion X to emotion Y, does anticipation
scale with how far apart X and Y sit in valence-arousal space? And does it do
so independently of which designed family the pair came from?**

**Method, plainly.** Family labels bundle several things at once, and the
registration names this cut as the way to separate the affective-distance
construct from the family label. We relate each transition's R1 lead
(`selfgen` probes, Gemma stories) to the NRC VAD (v2.1) geometry of its
from-to emotion pair in two cuts. First, the lead binned by absolute valence
difference. Second, the headline contrast: cross-valence transitions, where
the pair straddles the pleasant-unpleasant boundary, against same-valence
transitions. The cell prints the per-layer correlations of the lead with the
valence gap and with the arousal gap below the figure.

In [4]:
# this cell relates each transition's R1 lead (self-generated probes, primary arm)
# to the VAD geometry of its from->to pair
fig, S3 = tr.s3_affective_distance_figure(ARMS, VAD, RNG)
fig.show()
print("pearson r(lead, |dV|) and r(lead, |dA|) per layer:")
for layer in LAYERS:
    print(f"  L{layer}: r_dval={S3[layer]['r_dval']:+.3f}  r_darr={S3[layer]['r_darr']:+.3f}")

pearson r(lead, |dV|) and r(lead, |dA|) per layer:
  L6: r_dval=+0.027  r_darr=-0.017
  L15: r_dval=+0.061  r_darr=-0.025
  L24: r_dval=+0.143  r_darr=+0.026
  L33: r_dval=+0.001  r_darr=+0.075
  L42: r_dval=+0.186  r_darr=+0.010
  L51: r_dval=+0.255  r_darr=-0.019


<details><summary><b>How to read this figure</b></summary>

Left panel: one bar is one bin of transitions, grouped by the absolute NRC
VAD valence gap between the outgoing and incoming emotion. The bin edges are
0.0, 0.4, 0.8, 1.2 and 2.0 on the lexicon's valence scale. Bar height is the
mean R1 lead of the transitions in that bin, with the bin's n printed on the
bar. Right panel: two bars, the mean lead for cross-valence transitions (from
and to emotions on opposite sides of valence zero) vs same-valence
transitions.
Error bars in both panels are 95% cluster-bootstrap CIs over `triple_id`. The
slider scrubs the six extracted layers (6, 15, 24, 33, 42, 51), default 33,
and the title restates that layer's own verdict.
If anticipation follows affective distance, the left panel rises with the
valence gap and the right panel separates cross from same. Flat bars with
overlapping CIs mean the affective-distance construct does not drive
anticipation at that layer, whatever the family cut in S2 showed. The print
block under the figure gives, per layer, the Pearson correlation of the lead
with the absolute valence gap and with the absolute arousal gap. That is a
check on whether the relation strengthens or weakens with depth. The registration named
this cut as the confound separator: an S2 family effect without an S3 distance
effect points at family composition, not affective distance.

</details>

## S4. When the model gets it wrong, what wins instead?

**Question: when the tagged emotion's probe does not win a phase, does an
affective neighbour win (graceful degradation) or an arbitrary probe
(unstructured failure)?**

**Method, plainly.** For every phase (`selfgen` probes, Gemma stories) we
record the winning probe, the one with the highest phase-mean centered cosine,
against the tagged emotion. The full 12x12 tagged-by-winner matrix is the
confusion heatmap. Then, for the wrong-winner phases only, we compare the
winner's VAD distance to the target with the same distance when a uniformly
random wrong probe replaces the winner. That random draw is the
unstructured-failure reference. Winners much closer than that reference mean
the model degrades towards neighbours, a different behaviour from random
failure.

In [5]:
# this cell builds the confusion matrix and the winner-vs-shuffle VAD
# distance comparison, self-generated probes, primary arm
fig, S4 = tr.s4_confusion_figure(ARMS, VAD, RNG)
fig.show()
for layer in LAYERS:
    layer_stats = S4[layer]
    print(f"L{layer}: top-1 {layer_stats['top1_rate']:.2f}; "
          f"wrong-winner VAD dist {layer_stats['wrong_mean']:.2f} "
          f"vs shuffle {layer_stats['shuffle_mean']:.2f} (n_wrong={layer_stats['n_wrong']})")

L6: top-1 0.58; wrong-winner VAD dist 0.78 vs shuffle 1.12 (n_wrong=264)
L15: top-1 0.33; wrong-winner VAD dist 0.96 vs shuffle 1.12 (n_wrong=424)
L24: top-1 0.57; wrong-winner VAD dist 0.89 vs shuffle 1.07 (n_wrong=270)
L33: top-1 0.27; wrong-winner VAD dist 1.08 vs shuffle 1.22 (n_wrong=460)
L42: top-1 0.41; wrong-winner VAD dist 0.92 vs shuffle 1.15 (n_wrong=372)
L51: top-1 0.30; wrong-winner VAD dist 0.99 vs shuffle 1.24 (n_wrong=438)


<details><summary><b>How to read this figure</b></summary>

Left panel: one cell is P(winning probe = column emotion) among the phases
tagged with the row emotion, rows normalised to sum to 1, darker blue = more
often. The diagonal is each emotion's top-1 rate (this panel contains S1's
diagonal plus the full off-diagonal structure); rows are the ground-truth
tag, columns are what the probes reported. Right panel: two bars. The first is
the mean VAD distance (Euclidean in the valence-arousal plane) from the actual
wrong winner to the target. The second is the same mean when a uniformly
random wrong probe replaces each wrong winner. The count of wrong-winner
phases is printed on the bars, and the same means are printed per layer under
the figure. This pair carries no bootstrap CIs. The slider scrubs the six
extracted layers (6,
15, 24, 33, 42, 51), default 33, and the title restates that layer's own
verdict. A good (graceful) result puts off-diagonal mass in affectively
similar columns, and the actual-winners bar clearly below the
random-wrong-probe bar, meaning failures land on affective neighbours. Bars of
equal height would mean failures are unstructured. A column that is dark for
many rows is a probe that wins indiscriminately, an artefact of the probe set
rather than a model belief. Registered caveat: emotions appear in different
families and story contexts (the family x emotion-pool confound). A row's
confusion profile therefore partly reflects which emotions co-occur with it in
triples, not only representational similarity. S8 returns to this with the
probe-geometry predictor.

</details>

## S5. Position and non-affect cuts

**Question: does tracking depend on where in the story the transition sits,
and do triples that mix in a non-affect concept break the affective probes?**

**Method, plainly.** Two smaller registered cuts on the primary condition,
using the `selfgen` probes. First, we split the R1 lead by transition
position. The second boundary has more context behind it and sits nearer the
end of the story, so a position effect would be a confound to carry into any
transition-level measure. Second, we split the median rank of the tagged
emotion by the `has_nonaffect` triple flag. Those triples mix an emotion with
a non-affect concept, which should be harder for a purely affective set of
probes if the distractor competes.

In [6]:
# this cell splits the tagged probe's rank (read G) and the R1 lead by transition
# position and by the has_nonaffect triple flag, self-generated probes, primary arm
fig, S5 = tr.s5_position_nonaffect_figure(ARMS, HAS_NONAFFECT, RNG)
fig.show()

<details><summary><b>How to read this figure</b></summary>

Left panel: one bar per transition position, the story's first vs second
emotion boundary. Height is the mean R1 anticipation lead over the
transitions at that position, with n printed on the bar (higher is better).
Right panel: one bar per value of the has_nonaffect flag. Height is the
MEDIAN RANK of the tagged emotion over the phases in triples with that flag.
Lower is better on the right while higher is better on the left, because the
two panels deliberately use two different measures. Error bars are 95%
cluster-bootstrap CIs over `triple_id`. The slider scrubs the six extracted
layers (6, 15, 24, 33, 42, 51), default 33. A good robustness result keeps
both cuts flat within CI: anticipation is not an artefact of boundary
position, and non-affect distractor phases do not break identity tracking. A
clear left-panel separation would flag position as a confound for
transition-level measures. A much worse rank at has_nonaffect=True would mean
the affective probes lose the thread when a non-affect concept is in play.
Registered caveat: phase length varies by position by design, so a position
effect here is not automatically a context-size effect. The family x
emotion-pool confound applies to any cut on designed triples.

</details>

## Cross-condition check and takeaway

**Question: do the Part 1 slices sit on top of the registered cross-condition
verdict (Gemma-written stories track best, DeepSeek stories weaker, the
constant-emotion control near zero)?**

**Method, plainly.** We print the same two measures (median rank of the tagged
emotion and mean R1 lead, `selfgen` probes) for all five conditions at the
primary layer. The constant-emotion control is the sanity anchor: no emotion
changes there, so any anticipation structure it shows is a scene-change
artefact rather than tracking. The block then reprints the Part 1 headline
numbers (S1 best and worst emotions, S3 valence contrast, S4
winner-vs-shuffle distance) in one place.

In [7]:
# this cell prints the compact cross-arm table at the primary layer and a
# plain-language reading assembled from the S1-S5 numbers computed above
print("\n".join(tr.cross_arm_lines(ARMS, S1, S3, S4)))

=== primary layer 33, self-generated probes ===
it_v2                  median rank of tagged probe  3.0 (n=630)   R1 mean lead +0.0117 (n=417)
it                     median rank of tagged probe  3.0 (n=630)   R1 mean lead +0.0116 (n=417)
deepseek               median rank of tagged probe  5.0 (n=590)   R1 mean lead +0.0044 (n=389)
deepseek_constant      median rank of tagged probe  4.5 (n=314)   R1 mean lead +0.0005 (n=208)
base                   this arm has no self-generated probes (its probes come from the story corpus); that read lives in S6 and in notebook 10's factorial

S1 best-tracked emotions: loving, guilty, afraid | worst: inspired, calm, nervous
S3 cross-valence lead +0.0118 vs same-valence +0.0126
S4 wrong-winner VAD distance 1.08 vs shuffle 1.22


<details><summary><b>How to read this output</b></summary>

The first block gives one line per condition at the primary layer 33, using
the `selfgen` probes. Each line prints the median rank of the tagged emotion,
with its phase n. It then prints the mean R1
lead with its transition n. The rank measures identity: 1 is perfect, and
guessing sits near 6.5 for 12 probes. A positive lead means the incoming
emotion rises before the boundary.
If these records reproduce the registered E1 verdict, the two Gemma-story
conditions lead on both measures, the DeepSeek-story condition is weaker, and
the constant-emotion control sits near zero lead. The remaining lines pull
single headline numbers from the S1, S3 and S4 structures computed above. They
give the three best- and worst-tracked emotions, the cross-valence vs
same-valence mean leads, and the wrong-winner vs random-shuffle VAD distances.
Everything here is a point reading at one layer with one set of probes and no
CIs. Use the section figures above for uncertainty, and treat all of it as
exploratory.

</details>

## Part 1 verdict

*(the numbers live in the cell outputs above; this cell only states how to
weigh them)*

- **S1** ranks emotions by how reliably their probe wins on phases tagged with
  them. A spread there is a statement about probe quality times expression
  strength, not about the model alone.
- **S2/S3** together say whether difficulty follows the designed family or the
  underlying affective distance.
- **S4** distinguishes graceful degradation (neighbour confusions) from
  unstructured failure.
- Nothing here becomes a claim until it survives a falsification pass; a
  promising slice becomes a registered measurement first.

# Part 2. Expected vs reported: named confusions, timing, geometry (TREE Q3.H1.E3)

**Question (user, 2026-07-23).** Three numerical follow-ups to Part 1. What
does the model REPORT the emotion is when it disagrees with our tag (named
confusions, not just distances)? WHEN inside a phase does it disagree
(boundary lag vs stable relabeling, both defined in S7)? And does the geometry
of the probe vectors themselves predict where tracking is hard? Human
affective distance (NRC VAD) is the competing predictor for that third
question, so the residual is what is idiosyncratic to the model.

**Status: exploratory**, registered before the extended records existed
(TREE Q3.H1.E3, commit `6d3684e`): the measures, the two named failure modes
for S7, and the geometry-vs-VAD partial design all predate the data.

## S6. What does the model say the emotion is?

**Question: when the model disagrees with the tag, which emotion does it
report instead, named rather than measured as a distance?**

**Method, plainly.** For each expected (tagged) emotion, the model's "answer"
for a phase is the probe with the highest mean centered cosine over that
phase. Three tables name the top reported emotions with their rates. The first
reads the primary condition with the `selfgen` probes. The second reads the
same condition with the `deepseek` probes, asking whether two probe sources
tell the same story. The third is the BASE reader on the `corpus` probes,
restricted to the same twelve emotions, so all three tables share one emotion
set. A per-family block then shows what each designed family's wrong answers
look like, and a heatmap gives the base reader's full confusion structure.

In [8]:
# this cell prints the named confusion tables at the primary layer, plus the
# base-reader confusion heatmap with a layer slider
fig, S6 = tr.s6_named_confusions(ARMS)
print("\n".join(S6["lines"]))
fig.show()

=== it_v2 / probes from its own stories, layer 33: expected -> model reports (rate) ===
  happy      (n= 52) -> happy 31%, angry 21%, loving 10%
  inspired   (n= 48) -> guilty 33%, happy 31%, inspired 12%
  loving     (n= 53) -> loving 49%, happy 21%, sad 15%
  proud      (n= 53) -> guilty 38%, happy 25%, proud 13%
  calm       (n= 53) -> happy 34%, guilty 17%, afraid 17%
  desperate  (n= 53) -> desperate 32%, guilty 23%, happy 15%
  angry      (n= 53) -> happy 26%, guilty 19%, afraid 19%
  guilty     (n= 53) -> guilty 49%, happy 15%, afraid 15%
  sad        (n= 53) -> happy 32%, sad 30%, guilty 19%
  afraid     (n= 53) -> afraid 43%, happy 21%, angry 15%
  nervous    (n= 52) -> afraid 33%, angry 21%, happy 13%
  surprised  (n= 54) -> surprised 31%, guilty 24%, happy 15%
=== it_v2 / probes from DeepSeek stories, layer 33: expected -> model reports (rate) ===
  afraid     (n= 53) -> afraid 43%, nervous 23%, angry 19%
  angry      (n= 53) -> angry 40%, afraid 21%, nervous 15%
  calm     

<details><summary><b>How to read this output</b></summary>

Each table line reads: expected emotion (with its phase n), then the model's
most-reported emotions with the rate at which each won that emotion's phases.
The expected emotion leading its own line at a high rate is the success case,
and any other leading name is a named confusion. The three tables differ only
in reader and probe source. Two of them use the instruct reader, once with the
`selfgen` probes and once with the `deepseek` probes. The third uses the base
reader with its corpus-built probes, cut to the twelve designed emotions
A confusion that appears in all
three is therefore a property of the stories or the emotion pair. A confusion
that appears in a single table follows that set of probes or that reader. The
tables are fixed at the primary layer 33.
The per-family block lists each designed family's wrong-answer rate and the
names the model substitutes, the direct expected-vs-reported deliverable. The
heatmap below covers the base reader only. One cell is P(reported = column
emotion) among phases tagged with the row emotion, row-normalised, so the
diagonal is agreement. Its slider scrubs the six extracted layers (6, 15, 24,
33, 42, 51), default 33. A good result is a dark diagonal with off-diagonal
mass on plausible neighbours. A column dark across many rows is a probe that
wins indiscriminately, an artefact of the probe set rather than a model belief.
Caveats: reported rates ride on probe quality times expression strength (S1's
caveat), per-emotion n differs, and the family block inherits the family x
emotion-pool confound.

</details>

## S7. When is the model right: boundary lag or stable relabeling?

**Question: when the model disagrees with the tag, is it merely late (right
by the end of the phase) or does it consistently tell a different story?**

**Method, plainly.** We split each phase into token-count thirds and score
them with the same centered-cosine convention (the E3 extension to the
records). Comparing the winning probe in the FIRST third against the LAST
third separates the two failure modes named at registration.
**Boundary lag** means wrong at the start, right by the end: the model just
needs tokens after a transition, and it shows up as a large "converges" share.
**Stable relabeling** means wrong the whole way: the model holds its own
consistent opinion about the story's emotion, the idiosyncratic-belief case,
and it shows up as "never right".

In [9]:
# this cell classifies every phase by first-third vs last-third correctness
fig, S7 = tr.s7_thirds_figure(ARMS)
fig.show()
print("\n".join(S7["lines"]))

layer 33 per family (it_v2, probes from its own stories): converges vs never-right
  A_superposition      converges  17%   never right  60%   (n=126)
  B_conflict           converges  14%   never right  66%   (n=241)
  D_timescale          converges   9%   never right  78%   (n=69)
  E_arousal_mismatch   converges  22%   never right  71%   (n=51)
  F_valence_spread     converges  17%   never right  64%   (n=143)

dominant failure mode at layer 33 (it_v2): stable relabeling


<details><summary><b>How to read this output</b></summary>

In the figure, one bar is one of four phase classes on one set of stories. The
left panel holds the Gemma stories and the right panel the DeepSeek stories,
both scored with the `selfgen` probes. The four classes come from whether the
tagged probe wins the first and the last token-count third of the phase. Two
of them are "always right" (wins both) and "never right" (wins neither, the
stable-relabeling signature). Between them sit "converges" (loses the first
third, wins the last, the boundary-lag signature) and "loses it" (wins first,
loses last). Bar heights are shares of that panel's phases and sum to 1 per
panel, with the percentage printed on each bar. There are no error bars on
these shares. The slider scrubs the six extracted layers (6, 15, 24, 33, 42,
51), default 33, and the title restates that layer's own verdict. Reading the
modes: "converges" clearly above "never right" means disagreement is mostly
timing, and the model catches up after the boundary. "Never right" dominating
means the model stably relabels, believing the phase is about a different
emotion than the tag. The print block below fixes layer 33. It gives the
converges vs never-right split per designed family, then names the dominant
failure mode by comparing the two shares on the primary condition. Caveats:
thirds differ in token count across phases. A "win" is the top-1 criterion
only, so an emotion sitting at rank 2 throughout still counts as never right.
The family split inherits the family x emotion-pool confound.

</details>

## S8. When the tracker is wrong, whose "similar" predicts the mistake?

**Question: which ruler predicts WHICH wrong emotion wins?** Two rulers
compete. The model's own geometry says how close two probe directions are, by
cosine. Human affect ratings say how close two emotions sit in NRC
valence-arousal space.

The two rulers agree with each other substantially (the first printed line
quantifies it), so raw correlations cannot separate them. Partial
correlations can: each ruler's predictive power AFTER removing what it
shares with the other. Whatever survives only on the model's side is, by
construction, idiosyncratic to the model.

**Method, plainly.** We reconstruct the exact set of probes: unit contrast
probes, each set centered on its own emotion pool, verified against the
recorded label order. For each of the 132 ordered emotion pairs (target, wrong
answer) we count how often that wrong answer wins phases tagged with the
target. We then correlate those rates with each ruler and take the two
partials. The right panel makes the size concrete: mean confusion rate among
the quartile of pairs the model's probes rate most similar vs the
least-similar quartile. Two smaller sub-measures print above the figure. (a)
Crowding: does a probe with a high mean cosine to the other 11 track worse?
(c) Transitions: are switches between similar probes harder to anticipate,
since they leave less contrast to detect?

In [10]:
# this cell reconstructs the exact probe geometry and tests it as a
# difficulty predictor, with NRC VAD distance as the competing predictor
fig, S8 = tr.s8_geometry_figure(ARMS, VAD)
print("\n".join(S8["lines"]))
fig.show()

probe cos vs VAD distance over the 132 ordered pairs, layer 33: spearman +0.71 (correlated, hence the partials below)

(a) crowding vs tracking quality, per layer (spearman, n=12 emotions):
  L6: rho = -0.47 (p=0.121)  <- crowded probes track worse
  L15: rho = -0.27 (p=0.404)
  L24: rho = -0.25 (p=0.442)
  L33: rho = -0.31 (p=0.330)  <- crowded probes track worse
  L42: rho = -0.19 (p=0.564)
  L51: rho = -0.03 (p=0.931)

(b) which wrong answer wins (132 pairs, layer 33):
  confusion ~ probe cos          spearman +0.18
  confusion ~ VAD closeness      spearman +0.06
  confusion ~ probe cos | VAD    partial  +0.20   <- the model-idiosyncratic read
  confusion ~ VAD | probe cos    partial  -0.09
  concrete contrast: the model's most-similar pair quartile is confused 9.8% of the time vs 5.7% for the least-similar quartile (1.7x)

(c) transitions with a probe for both emotions (n=70, layer 33):
  R1 lead ~ cos(from, to)                spearman -0.17
  post-boundary tagged rank ~ cos       

<details><summary><b>How to read this figure</b></summary>

Left panel: four bars, one per way of predicting the confusions. Blue bars
use the model's ruler (probe cosine), green bars the human ruler (NRC
valence-arousal closeness). The first two are raw rank correlations; the
last two are partials, each ruler's power after removing the other. The
marked zero line means no predictive power; a good ruler stands clearly
above it, a bar near zero fails. The observed pattern at the primary layer:
the model's ruler keeps most of its height in partial form, while the human
ruler drops to about zero. So the model confuses what IT represents as
similar. Human similarity only appeared to matter because the two rulers
overlap.

Right panel: the same fact in concrete units. Bars are the average
probability that a wrong emotion wins a phase, for the quartile of pairs
the model's probes rate most similar vs the least-similar quartile. The
dotted line is the all-pairs average. A large gap says geometry matters in
practice, not just in rank order.

The printed block above the figure carries the two smaller sub-measures.
(a) Crowding gives one Spearman rho per layer over only 12 emotions, so read
it as directional. (c) Transitions correlates each transition's anticipation
lead and post-boundary rank with cos(from-probe, to-probe), n=70. The slider
recomputes both panels at each of the six extracted layers, default 33, the
registered primary. It restates the title with that layer's own verdict, which
is not always the same ruler. Registered caveats: the `selfgen` and `deepseek`
probes are centered on their own twelve-emotion pool. Cosines within a set are
therefore shifted negative, and only their ordering is meaningful. Pair-level
confusion rates also ride on modest per-emotion n, so treat small bar
differences as noise.

</details>

## Part 2 verdict

- **S6** is the numerical answer to "what does the model think it is": per
  expected emotion, the named report distribution; per family, the wrong-answer
  profile. Read it with S1's caveat (probe quality times expression strength).
- **S7** adjudicates between the two failure modes named at registration; the
  dominant mode is printed, per condition and per family.
- **S8** separates model-idiosyncratic geometry from human affective distance
  via the partial correlations; sub-measure (c) tests whether transitions
  between similar probes are harder.
- Same standing as Part 1: exploratory, and nothing becomes a claim until it
  survives a falsification pass.

**What the Part 2 results mean, the live hypotheses, and what would decide
them** (interpretation, not claims; the numbers live in the cell outputs
above):

- *The wrong answers pile onto a couple of probes.* Suppose a couple of probes
  win most wrong phases regardless of the target (see S6). The natural reading
  is then a defect in how we built those probes, not a model belief. Those
  directions may sit closer to generic story activity, so everything leans
  towards them.
  Deciding test: audit how those probes were built (per-story variance, norm
  before scaling to unit length, cosine to the mean of the set) and re-run S6
  with them removed. An artefact of our probes disappears; a genuine basin in
  the model's reading does not.
- *Stable relabeling.* If disagreement holds from a phase's first third to
  its last (S7), the tracker is not lagging the boundary. It holds its own
  consistent reading of the phase. That makes the disagreements themselves
  data. They are either the model's genuine opinion about the text or the
  probe artefact above. Test for the first: human-rate a sample of never-right
  phases. If raters side with the model, our tags are wrong, not the tracker.
- *Geometry over human distance.* If which wrong answer wins follows the
  model's own probe cosines even after removing the human valence-arousal
  ruler (S8), then confusion structure is model-idiosyncratic. Any future
  claim that the model confuses affectively similar emotions must then be
  registered against probe geometry, not against a human lexicon. Deciding
  test, to run before such a claim could survive a falsification pass:
  replicate the partial-correlation ordering on the `deepseek` probes and the
  base-reader condition, where the probe geometry differs.